In [1]:
import os

In [2]:
%pwd

'c:\\Users\\pm062\\Desktop\\End_to_End_TEXT_SUMMARIZER\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\pm062\\Desktop\\End_to_End_TEXT_SUMMARIZER'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [6]:
from text_summarizer.constants import *
from text_summarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path = config.model_path,
            tokenizer_path = config.tokenizer_path,
            metric_file_name = config.metric_file_name
           
        )

        return model_evaluation_config

In [8]:
import torch
import pandas as pd
from tqdm import tqdm
import evaluate

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_from_disk
from datasets import load_dataset, load_metric

c:\Users\pm062\Desktop\End_to_End_TEXT_SUMMARIZER\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-07-11 18:14:29,451: INFO: config: PyTorch version 2.13.0 available.]


In [9]:
class ModelEvaluation:

    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """
        Split the dataset into smaller batches.
        """
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i:i + batch_size]

    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=16,
        device="cuda" if torch.cuda.is_available() else "cpu",
        column_text="dialogue",
        column_summary="summary"
    ):

        article_batches = list(
            self.generate_batch_sized_chunks(
                dataset[column_text], batch_size
            )
        )

        target_batches = list(
            self.generate_batch_sized_chunks(
                dataset[column_summary], batch_size
            )
        )

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=len(article_batches)
        ):

            inputs = tokenizer(
                article_batch,
                max_length=1024,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                length_penalty=0.8,
                num_beams=8,
                max_length=128
            )

            decoded_summaries = [
                tokenizer.decode(
                    summary,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True
                )
                for summary in summaries
            ]

            metric.add_batch(
                predictions=decoded_summaries,
                references=target_batch
            )

        score = metric.compute()
        return score

    def evaluate(self):

        device = "cuda" if torch.cuda.is_available() else "cpu"

        print(f"Using device: {device}")
        print(f"Loading dataset from: {self.config.data_path}")

        tokenizer = AutoTokenizer.from_pretrained(
            self.config.tokenizer_path
        )

        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_path
        ).to(device)

        # Load transformed dataset
        dataset_samsum_pt = load_from_disk(
            self.config.data_path
        )

        rouge_metric = evaluate.load("rouge")

        score = self.calculate_metric_on_test_ds(
            dataset=dataset_samsum_pt["test"].select(range(10)),
            metric=rouge_metric,
            model=model_pegasus,
            tokenizer=tokenizer,
            batch_size=2,
            column_text="dialogue",
            column_summary="summary"
        )

        rouge_dict = {
            "rouge1": score["rouge1"],
            "rouge2": score["rouge2"],
            "rougeL": score["rougeL"],
            "rougeLsum": score["rougeLsum"]
        }

        df = pd.DataFrame(
            [rouge_dict],
            index=["pegasus"]
        )

        df.to_csv(
            self.config.metric_file_name,
            index=False
        )

        print("ROUGE Scores:")
        print(df)

In [10]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluate()
except Exception as e:
    raise e

[2026-07-11 18:14:34,722: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-11 18:14:34,727: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-11 18:14:34,728: INFO: common: created directory at:artifacts]
[2026-07-11 18:14:34,728: INFO: common: created directory at:artifacts/model_evaluation]
Using device: cpu
Loading dataset from: artifacts/data_transformation/samsum_dataset


100%|██████████| 5/5 [09:45<00:00, 117.07s/it]


[2026-07-11 18:24:33,864: INFO: rouge_scorer: Using default tokenizer.]
ROUGE Scores:
          rouge1   rouge2    rougeL  rougeLsum
pegasus  0.26583  0.06327  0.197876   0.197822


In [11]:
print(model_evaluation_config.config.data_path)

artifacts/data_transformation/samsum_dataset


In [12]:
import os

path = "artifacts/data_transformation/samsum_dataset"

print("Exists:", os.path.exists(path))
print("Contents:", os.listdir(path))

Exists: True
Contents: ['dataset_dict.json', 'test', 'train', 'validation']


In [13]:
print(os.listdir("artifacts/data_transformation/samsum_dataset/train"))

['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']


In [14]:
from datasets import load_from_disk

dataset = load_from_disk("artifacts/data_ingestion/samsum_dataset")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})


In [15]:
from datasets import load_from_disk

dataset = load_from_disk("artifacts/data_transformation/samsum_dataset")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
})
